# Lesson 5, Part 2

In [ ]:
# Initializations
import numpy as np
import matplotlib.pyplot as plt

def eval_poly(x, coef):
    poly = coef[-1]
    for c in coef[-2::-1]:
        poly = poly * x + c
    return poly

The code we developed to find zeros of polynomials is easily extended to cover other functions. 
Here's an extension for the secant method.

In [ ]:
def secant_zero(x0,x1,fun,tol=1E-14):
    y0,y1 = fun(x0),fun(x1)
    for n in range(100):
        x2 = x0 - y0 * (x1 - x0) / (y1 - y0)
        y2 = fun(x2)
        if abs(y2) < tol: 
            break 
        if y2 != y1:
            x0,y0 = x1,y1
        x1,y1 = x2,y2
    return x2

# demo
coef = np.array([-20,0,0,1])
x1,x2 = 0,5

def fun1(x):
    return eval_poly(x,coef)
print('fun1 has zero at',secant_zero(x1,x2,fun1))

def fun2(x): 
    return np.sqrt(25 - x**2) - x
print('fun2 has zero at', secant_zero(x1,x2,fun2))

This extension looks a bit messy. 
Compressing definitions into single lines can save space. 
A simpler definition uses something Python calls a **lambda** function. 
Here are some illustrations.

In [ ]:
def fun1a(x): return eval_poly(x,coef)
fun1b = lambda x: eval_poly(x,coef)

def fun2a(x): return np.sqrt(25 - x**2) - x
fun2b = lambda x: np.sqrt(25 - x**2) - x

coef = np.array([-20,0,0,1])
y = np.linspace(0,5,100)
print('Do fun1, fun1a, and fun1b do the same thing?',np.all(fun1a(y)==fun1(y)) and np.all(fun1b(y)==fun1(y)))
print('Do fun2, fun2a, and fun2b do the same thing?',np.all(fun2a(y)==fun2(y)) and np.all(fun2b(y)==fun2(y)))
x1,x2 = 0,5
print('The function x**3-27 has a zero at', secant_zero(x1,x2,lambda x: x**3 - 27))

The lambda form dispenses with the **def** and **return**, which makes for simpler reading. 
Sometimes we don't even need to name it. 
Lambda functions also allow for multiple inputs.

In [ ]:
y_circle = lambda radius,x: np.sqrt(radius**2 - x**2)
radius,x = 5,4
print('For circle centered at origin with radius =',radius,'and x =', x,', |y| =',y_circle(radius,x))

Lambda functions are limited to single expressions. 
However, we can get around that by stacking lambda functions.

In [ ]:
# fun = lambda x: y = 25 - x**2, np.sqrt(y) - x

# this version will work
fun1 = lambda x: 25 - x**2
fun2 = lambda x: np.sqrt(fun1(x)) - x
print('fun1(5) =',fun1(5),', fun2(5) =',fun2(5))

# two ways to define the derivative of a polynomial
deriv_coef = lambda coef: [0] if len(coef)==0 else coef[1:] * np.arange(1,len(coef))
deriv_poly = lambda x,c: eval_poly(x, deriv_coef(c))  
x = np.linspace(1,4,4)
coef = [-10,0,0,1]
print('coef =',coef,', deriv_coef =',deriv_coef(coef))
print('x,coef =',x,coef,', deriv_poly =',deriv_poly(x,coef))

Here is a Newton-Raphson algorithm that uses a lambda function 
to substitute for missing info on derivatives.

In [ ]:
def newton_raphson(x0,fun,deriv,tol=1E-14):
    x = x0
    if deriv==None:
        deriv = lambda z: 1000 * (fun(z+0.001) - fun(z)) 
    for n in range(100):
        y = fun(x)
        if abs(y) < tol:
            break
        x -= y/deriv(x)
    return x

coef = [-20,0,0,2]
poly = lambda x: eval_poly(x,coef)
deriv = lambda x: deriv_poly(x,coef)
print('zero for fun1 at',newton_raphson(1,poly,deriv))
print('zero for fun1 at',newton_raphson(1,poly,None))
fun2 = lambda x: np.sqrt(25 - x**2) - x
deriv_fun2 = lambda x: -x/np.sqrt(25-x**2) -1
print('\nzero for fun2 at',newton_raphson(1,fun2,deriv_fun2))
print('zero for fun2 at',newton_raphson(1,fun2,None))

Halley's method uses the second derivative to fine-tune the Newton-Raphson adjustments.

In [ ]:
def halley(x0,fun,deriv1,deriv2,tol=1E-14):
    x = x0
    if deriv1==None:
        deriv1 = lambda z: 1000 * (fun(z+0.001) - fun(z))
    if deriv2==None:
        deriv2 = lambda z: 100 * (deriv1(z+0.01) - deriv1(z))
    for n in range(100):
        y = fun(x)
        if abs(y) < tol:
            break
        dydx = deriv1(x)
        d2ydx2 = deriv2(x)
        numer = 2 * y * dydx
        denom = 2 * dydx**2 - y * d2ydx2
        x -= numer / denom
    return x
    
coef = [-20,0,0,2]
poly = lambda x: eval_poly(x,coef)
deriv1 = lambda x: deriv_poly(x,coef)
deriv2_poly = lambda x,c: eval_poly(x, deriv_coef(deriv_coef(c)))
deriv2 = lambda x: eval_poly(x, deriv_coef(coef)) 

print('zero for polynomial with new coefficients at',halley(1,poly,deriv1,deriv2))
print('zero for polynomial with new coefficients at',halley(1,poly,deriv1,None))
print('zero for polynomial with new coefficients at',halley(1,poly,None,None))

Let's compare the speed of different zero-finders. 
While the results vary with the function and the initial seed, 
a few examples reveal some general patterns. 
Not surprisingly, the secant method is the most sensitive to the initial seeds, 
and using exact formulas for derivatives is typically quicker than estimating them from scratch.

In [ ]:
from time import time as T

coef = [-20,0,0,1]
coef_deriv1 = deriv_coef(coef)
coef_deriv2 = deriv_coef(coef_deriv1)
poly = lambda x: eval_poly(x,coef)
deriv1 = lambda x: eval_poly(x,coef_deriv1)
deriv2 = lambda x: eval_poly(x,coef_deriv2)

iters = 1000; tic = T() 
for iter in range(iters): secant_zero(0,10,poly)
print('secant method w x0,x1=0,10:', T() - tic); tic = T() 
for iter in range(iters): secant_zero(0,5000,poly)
print('secant method w x0,x1=0,5000:', T() - tic); tic = T()  
for iter in range(iters): newton_raphson(1,poly,deriv1)
print('Newton-Raphson w x0=1:', T() - tic); tic = T() 
for iter in range(iters): newton_raphson(1,poly,deriv1)
print('Newton-Raphson w x0=1, no deriv:', T() - tic); tic = T()
for iter in range(iters): newton_raphson(500,poly,deriv1)
print('Newton-Raphson w x0=500:', T() - tic); tic = T()
for iter in range(iters): newton_raphson(500,poly,deriv1)
print('Newton-Raphson w x0=500, no deriv:', T() - tic); tic = T() 
for iter in range(iters): halley(1,poly,deriv1,deriv2)
print('Halley method w x0=1:', T() - tic); tic = T()
for iter in range(iters): halley(1,poly,deriv1,None)
print('Halley method w x0=1, no deriv1:', T() - tic); tic = T()
for iter in range(iters): halley(1,poly,None,None)
print('Halley method w x0=1, no deriv1 or 2:', T() - tic); tic = T()
for iter in range(iters): halley(500,poly,deriv1,deriv2)
print('Halley method w x0=500:', T() - tic); tic = T()
for iter in range(iters): halley(500,poly,deriv1,None)
print('Halley method w x0=500, no deriv1:', T() - tic); tic = T()
for iter in range(iters): halley(500,poly,None,None)
print('Halley method w x0=500, no deriv1 or 2:', T() - tic)

Let's turn to Taylor polynomials. The k-th derivative of (*x - a*)<sup>*k*</sup> at equals k! = k(k-1)(k-2)...(2)1. 
This is known as a **factorial**. 
It can be computed in various ways but the fastest in Python uses a built-in command from the **math** module.

In [ ]:
from math import factorial

def factorial1(n):
    result = 1
    for i in range(2,n+1):
        result *= i
    return result

def factorial2(n):
    result = n
    for i in reversed(range(2,n)):  # reversed() reverses order of iteration
        result *= i
    return result

def factorial3(n):                  # recursive function
    return 1 if n==1 else n * factorial3(n-1)


# demo
n = 20
print('factorial of',n,'=',factorial(n),factorial1(n),factorial2(n),factorial3(n))
iters = 50000; tic = T() 
for iter in range(iters): factorial(n)
print("factorial  takes time", T() - tic); tic = T() 
for iter in range(iters): factorial1(n)
print("factorial1 takes time", T() - tic); tic = T() 
for iter in range(iters): factorial2(n)
print("factorial2 takes time", T() - tic); tic = T() 
for iter in range(iters): factorial3(n)
print("factorial3 takes time", T() - tic); tic = T()

The Taylor polynomial of order *m*  matches a function *f* 
and its first *m* derivatives at some point *a*. 
Extra Taylor terms are easily appended by using previous terms as basis, 
without explicitly calculating factorials. 
Here is an example for fractional power functions.

In [ ]:
# generate Taylor series coefficients for power function
def taylor_power_fun(a,coef,power,order):
    taylor_coef = [coef * a**power]
    for k in range(1,order+1):
        coef,power = coef*power,power-1
        taylor_coef.append(coef/factorial(k))
    return taylor_coef

# same results but faster
def taylor_power_faster(a,coef,power,order):
    taylor_coef = [coef * a**power]
    for k in range(1,order+1):
        coef *= power/k
        power -= 1
        taylor_coef.append(coef)
    return taylor_coef

a,coef,power,order = 2,1,0.5,8
iters = 20000
for fn in taylor_power_fun, taylor_power_faster:
    print('Power',power,', coefficients of',fn.__name__)  # .__name__  gives name of function
    print(fn(a,coef,power,order))
    tic = T()
    for iter in range(iters):
        fn(a,coef,power,order)
    print(iters,'iterations take time',T()-tic,'\n')

For points near *a*, the Taylor polynomial of order *m* approximates *f* well, 
and the higher *m* the better.
However, the fit may be poor for points farther away and get worse as *m* increases.
Here is an example for *f*(*x*) = sqrt(*x*) near *x*=1.

In [ ]:
taylor = lambda z,taylor_coef: eval_poly(z,taylor_coef)

a,coef,power = 1,1,0.5
x = np.linspace(0,4,400)
y = coef * x**power
plt.title('Approximation of x**' + str(power) +' with Taylor series order 2-4')
plt.plot(x,y,label='x**'+str(power))
for order in [2,3,4]:
    taylor_coef = taylor_power_faster(a,coef,power,order)
    z = taylor(x-a,taylor_coef)
    plt.plot(x,z,label='Taylor order '+str(order))
plt.legend()
plt.show()

In this example, raising *m* improves the fit for *x<*2 but worsens the fit for *x>*2.

In [ ]:
a,coef,power = 1,1,0.5
x = np.linspace(0,2.5,400)
y = coef * x**power
plt.title('Approximation of x**' + str(power) +' with higher-order Taylor series')
for order in [4,7,10,13,16]:
    taylor_coef = taylor_power_faster(a,coef,power,order)
    z = taylor(x-a,taylor_coef)
    plt.plot(x,z,label='Taylor order '+str(order))
plt.plot(x,y,label='x**'+str(power))
plt.legend()
# closeup for shorter interval
x = np.linspace(0.1,2.26,400)
y = coef * x**power
plt.figure()
plt.title('Errors in approximation')
for order in [4,7,10,13,16]:
    taylor_coef = taylor_power_faster(a,coef,power,order)
    z = taylor(x-a,taylor_coef) - y
    plt.plot(x,z,label='Taylor order '+str(order))
plt.legend()
plt.show()

Taylor polynomials also help with numerical integration. 
They indicate that Simpson's rule, which assigns relative weights of (1,4,2,4,...,2,4,1)
to equally spaced *x*, approximates integrals better than the trapezoidal rule we used earlier,
with weights (1,2,2,...,2,2,1). Let's confirm this for a few examples.

In [ ]:
def trapezoidal(xmin,xmax,fun,num=100):
    x = np.linspace(xmin,xmax,num)
    dx = (xmax - xmin) / (num-1)
    y = fun(x)
    wtd_sum_y = np.sum(y) - 0.5 * y[0] - 0.5 * y[-1] 
    return wtd_sum_y * dx

def simpson(xmin,xmax,fun,num=100):
    num = 2 * (num//2) + 1  # need odd number of points
    x = np.linspace(xmin,xmax,num)
    dx = (xmax - xmin) / (num-1)
    y = fun(x)
    weight = np.zeros(num)
    weight[0] = weight[-1] = 1/3
    weight[1:-1:2] = 4/3
    weight[2:-2:2] = 2/3
    wtd_sum_y = np.sum(y * weight)
    return wtd_sum_y * dx

xmin,xmax,num = 0,5,101
print('Integration of x**power between',xmin,'and',xmax,'using values at',num,'points')
for power in range(3,11):
    fun = lambda x: x**power
    int_true = xmax**(power+1) / (power+1)
    dev_trap = trapezoidal(xmin,xmax,fun,num) / int_true - 1
    dev_simp = simpson(xmin,xmax,fun,num) / int_true - 1
    print('power = ',power,': relative error',round(dev_trap,4),'for trapezoidal and',round(dev_simp,9),'for simpson')

In fact, Simpson's rule is so good that we don't need many points for good estimation.

In [ ]:
xmin,xmax,num = 0,5,11
print('Integration of x**power between',xmin,'and',xmax,'using values at',num,'points')
for power in range(3,11):
    fun = lambda x: x**power
    int_true = xmax**(power+1) / (power+1)
    dev_trap = trapezoidal(xmin,xmax,fun,num) / int_true - 1
    dev_simp = simpson(xmin,xmax,fun,num) / int_true - 1
    print('power = ',power,': relative error',round(dev_trap,3),'for trapezoidal and',round(dev_simp,5),'for simpson')

Despite the benefits of Simpson's rule with smooth functions, 
trapezoidal approximations are more common and numpy provides a function **trapezoid** to compute them. 
Its main arguments are (*y,x*={None},*dx*=1), where *y*=*f*(*x*). 
That is, **trapezoid**(*y*) sets *dx*=1, **trapezoid**(*y,dx=val*) sets *dx=val*, and **trapezoid**(*y,x*) computes *dx* from *x*.

In [ ]:
print('Difference between trapezoid variants and trapezoidal:')
xmin,xmax,num = 0,10,101
dx = (xmax - xmin) / (num - 1)
x = np.linspace(xmin,xmax,num)
for power in range(3,11):
    fun = lambda x: x**power
    area1 = trapezoidal(xmin,xmax,fun,num)
    area2 = np.trapezoid(fun(x),x)
    area3 = np.trapezoid(fun(x),None,dx)
    print('power =',power,':',round(area2-area1,12),'or',round(area3-area1,12))